# E08: Geometrías, unión tabular e intersección

[Enunciado](../../talleres/E08.md) · [Entorno WSL](../../docs/instalacion-wsl.md)

Ejecutar en orden con el kernel del curso. Preparar las fuentes indicadas en el enunciado; no se descargan automáticamente. Las salidas se regeneran en `kit/salidas/`. Los originales se conservan.

In [ ]:
from pathlib import Path
import sys, json
import pandas as pd
import geopandas as gpd
from IPython.display import display, Image
candidatos = [Path.cwd(), *Path.cwd().parents, Path.cwd() / 'big-data-postgraduate']
REPO = next((p for p in candidatos if (p/'kit/talleres.py').exists()), None)
assert REPO is not None, 'Abrir desde el repositorio o su carpeta base bigdata'
KIT = REPO/'kit'
sys.path.insert(0, str(KIT))
import talleres as t
import geoespacial as geo
RAW, OUT = t.RAW, t.OUT


## 1. Leer y revisar CRS
Los ZIP conservan los componentes del Shapefile. No asignar un CRS para ocultar un metadato ausente. Transformar con `to_crs` cuando corresponde.

In [ ]:
municipios=geo.leer_vector('dane_municipios.zip')
departamentos=geo.leer_vector('dane_departamentos.zip')
print('CRS:',municipios.crs,departamentos.crs)
assert len(municipios)==1122
campo=next(c for c in municipios if c.lower()=='mpio_cdpmp')
municipios['codigo_municipio']=municipios[campo].astype('string').str.zfill(5)
assert municipios.codigo_municipio.is_unique
display(municipios[['codigo_municipio','geometry']].head())

## 2. Preparar una tabla municipal
Se usa AGROSAVIA con DIVIPOLA: se concilian departamento y municipio antes de agrupar. La cardinalidad territorial se comprueba y las filas sin código se reportan, sin asignarles ubicación inventada.

In [ ]:
datos=t.load('agrosavia.csv')
dim=t.load('divipola.csv')
dim['codigo_municipio']=dim['Código Municipio'].str.zfill(5)
dim['clave']=dim['Nombre Departamento'].map(t.norm)+'|'+dim['Nombre Municipio'].map(t.norm)
datos['clave']=datos['Departamento'].map(t.norm)+'|'+datos['Municipio'].map(t.norm)
datos=datos.merge(dim[['clave','codigo_municipio']],how='left',on='clave',validate='many_to_one')
datos['anio']=pd.to_datetime(datos['Fecha de Análisis'],format='%d/%m/%Y',errors='coerce').dt.year
print('Sin código territorial:',int(datos.codigo_municipio.isna().sum()))
print('Sin año interpretable:',int(datos.anio.isna().sum()))
anio=int(datos.anio.max())
tabla=datos.loc[datos.anio.eq(anio)].groupby('codigo_municipio').size().rename('registros').reset_index()
mapa=municipios.merge(tabla,how='left',on='codigo_municipio',validate='one_to_one',indicator=True)
assert len(mapa)==len(municipios)
print('Año:',anio,'municipios sin registros en la tabla:',int(mapa._merge.eq('left_only').sum()))
print('Códigos de la tabla sin geometría:',tabla.loc[~tabla.codigo_municipio.isin(municipios.codigo_municipio),'codigo_municipio'].tolist())
mapa.drop(columns='geometry').to_csv(OUT/'union_municipal_agrosavia.csv',index=False)

Los conteos corresponden a registros del archivo en el año elegido, no a cobertura de suelo ni a una muestra representativa. Los municipios sin correspondencia se muestran como faltantes, sin convertirlos automáticamente a cero.

In [ ]:
import matplotlib.pyplot as plt
fig,ax=plt.subplots(figsize=(9,8))
mapa.plot(column='registros',legend=True,missing_kwds={'color':'lightgrey'},ax=ax)
limites=ax.axis()
departamentos.to_crs(mapa.crs).boundary.plot(ax=ax,color='black',linewidth=.3)
ax.axis(limites)
ax.set_title(f'Registros AGROSAVIA, {anio}; gris: sin registros unidos')
fig.savefig(OUT/'mapa_agrosavia.png',dpi=160,bbox_inches='tight');plt.close(fig)
display(Image(filename=str(OUT/'mapa_agrosavia.png')))

## 3. Intersectar las dos muestras IGAC
El módulo usa `gpd.overlay`, registra reparaciones y mide hectáreas con `to_crs(9377).area / 10000`. El recorte se conserva en WGS84 para poder comparar con la implementación anterior. Solo se conservan intersecciones de área positiva.

In [ ]:
resultado=geo.intersecciones()
assert len(resultado)==15
assert resultado.area_interseccion_ha.gt(0).all()
display(resultado[['tema','codigo_municipio','area_interseccion_ha']])
display(json.loads((OUT/'control_geografico.json').read_text()))
display(Image(filename=str(OUT/'mapa_intersecciones.png')))

## 4. Verificar el GeoPackage y los atributos
GeoPackage conserva nombres largos. Los atributos de área originales no se recalculan al recortar: compararlos con `area_interseccion_ha` solo después de revisar sus unidades. Nunca usar estas muestras como cobertura municipal completa.

In [ ]:
recuperado=gpd.read_file(OUT/'intersecciones_muestra.gpkg',layer='intersecciones')
assert len(recuperado)==len(resultado) and recuperado.crs==resultado.crs
assert set(recuperado.columns)==set(resultado.columns)
display(resultado[[c for c in resultado.columns if 'area' in c.lower()]].head())
print('GeoPackage verificado; columnas:',resultado.columns.tolist())